# Test access to Google Gemini 3 pro - Image
https://console.cloud.google.com/vertex-ai/publishers/google/model-garden/gemini-3-pro-preview

Gemini 3 Pro is designed to tackle the most challenging agentic problems with strong coding and state-of-the-art reasoning capabilities. It is the **best model for complex multimodal understanding**. Compared to Gemini 2.5 Pro, it improves significantly on complex instruction following and delivers outcomes with better output efficiency.

Gemini 3.0 models are thinking models, capable of reasoning through their thoughts before responding, resulting in enhanced performance and improved accuracy.

The Gemini 3 Pro Preview model is now available on Vertex AI.

Gemini 3 Pro Preview is also available through our Gen AI SDK that provides a unified interface for Google AI Studio and Vertex AI in Python and Go.



## pre-requisistes

- gcloud console tool (as this will manage authentication)
  - for CURL calls:
    > gcloud auth login
  - for SDK calls:
    > gcloud auth application-default login
- enable model in model hub



## login
(only required once per session + zscaler security should be turned off)

In [ ]:
!gcloud auth application-default login

## config

In [ ]:
# Add all available artwork files here
# ARTWORK_FILE = "1948156_1419323_SIM_2"
# ARTWORK_FILE = "1991965_1825900_SIM"
# ARTWORK_FILE = "2741283_1961801_SIM"
# ARTWORK_FILE = "3019232_2965651_SIM"
# ARTWORK_FILE = "30511509_2"
# ARTWORK_FILE = "30633405_4"
ARTWORK_FILE = "30652435_1"
# ARTWORK_FILE = "30660179_2"
# ARTWORK_FILE = "30661064_2"
# ARTWORK_FILE = "30696442_2"
# ARTWORK_FILE = "30690350_1"
# ARTWORK_FILE = "30698797_1"
# ARTWORK_FILE = "30698853_2"
# ARTWORK_FILE = "30698938_1"
# ARTWORK_FILE = "30698973_1"
# ARTWORK_FILE = "30699352_1"

# Active file
#ARTWORK_FILE = "30678229_1"


DPI = 300
FILE_PATH_IN = "../data/in"
FILE_PATH_OUT = "../data/out"

In [ ]:
PROJECT_ID="hmcp-tst-veraitst-de-prj-arg"
MODEL="gemini-3-pro-preview"
LOCATION="global"

## load required libraries

In [ ]:
from IPython.display import JSON, Image, Markdown, display
from google import genai
from google.genai import types
import fitz
import base64
from PIL import Image as PIL_Image, ImageDraw as PIL_ImageDraw
import matplotlib.pyplot as plt
import json
import base64
import os
#from weasyprint import HTML
import markdown
from pydantic import BaseModel, Field
from enum import Enum
from typing import List, Optional
from io import BytesIO

## load artwork

In [ ]:
file_path = os.path.join(FILE_PATH_IN,ARTWORK_FILE+".pdf")
doc = fitz.open(file_path)
for page_num in range(len(doc)):
    page = doc.load_page(page_num)
    zoom = DPI / 72  # convert DPI to zoom factor
    matrix = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=matrix)

    image_output_path = f"{FILE_PATH_OUT}/{ARTWORK_FILE}_page_{page_num + 1}.jpg"
    pix.save(image_output_path)
    print(f"Saved: {image_output_path}")

In [ ]:
PIL_Image.MAX_IMAGE_PIXELS = None   # disables the safety limit of Pillow (DecompressionBombError)
img = PIL_Image.open(image_output_path)
width, height = img.size
display (img)

In [ ]:
with open(image_output_path, "rb") as artwork:
    artwork_b64 = base64.b64encode(artwork.read()).decode("utf-8")
    
artwork_inline_data = {
    "inline_data": {
         "mime_type": "image/jpeg",
        "data": artwork_b64
    }
}

## Connect to Gemini

In [ ]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

In [ ]:
PROJECT_ID="hmcp-tst-veraitst-de-prj-arg"
MODEL="gemini-3-pro-preview"
IMAGE_MODEL="gemini-3-pro-preview"
LOCATION="global"

## Step 1.5: DPI Calibration
Identify measurement lines to calculate true DPI.

In [ ]:
class Point(BaseModel):
    x: int
    y: int

class MeasurementLine(BaseModel):
    start_point: Point
    end_point: Point
    value_mm: float = Field(description="The numeric value in mm associated with the line")

class CalibrationCheck(BaseModel):
    measurement_line: Optional[MeasurementLine] = None

PROMPT_CALIBRATION = """
Identify a technical measurement line (ruler/dimension) on this artwork that is labeled in 'mm'. 
Return the start and end pixel coordinates of the line and its numeric value.
If no such line is found, return null.
"""

response = client.models.generate_content(
  model=MODEL,
  contents=[
    PROMPT_CALIBRATION,
    artwork_inline_data
  ],
   config={     
    "response_mime_type": "application/json",
    "response_json_schema": CalibrationCheck.model_json_schema(),
    }   
)

calibration_data = CalibrationCheck.model_validate_json(response.text)
display(JSON(calibration_data.model_dump()))

TRUE_DPI = DPI # Default fallback
DPmm = DPI / 25.4

if calibration_data.measurement_line:
    line = calibration_data.measurement_line
    # Calculate pixel length
    px_length = ((line.end_point.x - line.start_point.x)**2 + (line.end_point.y - line.start_point.y)**2)**0.5
    
    # Calculate DPmm and DPI
    if line.value_mm > 0:
        calculated_dpmm = px_length / line.value_mm
        calculated_dpi = calculated_dpmm * 25.4
        
        print(f"Found measurement line: {line.value_mm} mm = {px_length:.2f} pixels")
        print(f"Calculated Resolution: {calculated_dpi:.2f} DPI ({calculated_dpmm:.2f} pixels/mm)")
        
        TRUE_DPI = int(round(calculated_dpi))
        DPmm = calculated_dpmm
    else:
        print("Invalid measurement value found. Using default DPI.")
else:
    print("No measurement line found. Using default DPI.")

print(f"Using Reference Resolution: {TRUE_DPI} DPI")

## Step 1: Identify CLP and Marketing parts

In [ ]:
PROMPT1 = f"""

You are an AI assistant that checks images of product labels for compliance with accessibility guidelines.
Identify which parts of the image includes ingredients list and a warnings section marked by hazard symbols and classify them as CLP-Parts
Identify which parts of the image includes marketing text or usage instructions and classify them as Non-CLP-Parts
Add for each part a useful label.
"""

In [ ]:
class Rectangle(BaseModel):
    ymin: int
    xmin: int
    ymax: int
    xmax: int

class PartClassification(str, Enum):
    CLP = "CLP"
    NON_CLP = "NON-CLP"

class Part(BaseModel):
    classification: PartClassification
    label: str
    rect: Rectangle = Field(
        default=None,
        description="Location on the image where the control has been identifed"
    )

class VisualCheck(BaseModel):
    parts: List[Part]

In [ ]:
response = client.models.generate_content(
  model=MODEL,
  contents=[
    PROMPT1,
    artwork_inline_data
  ],
   config={     
    "response_mime_type": "application/json",
    "response_json_schema": VisualCheck.model_json_schema(),
    }   
)

In [ ]:
visual_check = VisualCheck.model_validate_json(response.text)
display(JSON(visual_check.model_dump()))

In [ ]:
draw = PIL_ImageDraw.Draw(img)

for part in visual_check.parts:
    rect = rect = part.rect
   
    abs_x1 = int(rect.xmin/1000 * width)
    abs_y1 = int(rect.ymin/1000 * height)
    abs_x2 = int(rect.xmax/1000 * width)
    abs_y2 = int(rect.ymax/1000 * height)

    draw.rectangle([(abs_x1, abs_y1), (abs_x2, abs_y2)], outline="red", width=30)
   
display(img)

## Step 2: Check parts

In [ ]:
PROMPT_CLP = f"""
You are an AI assistant that checks images of product labels for compliance with accessibility guidelines.
### **Reference Scale:**
The image resolution is calculated as {TRUE_DPI} DPI ({DPmm:.2f} pixels per mm). Use this to verify font sizes.

### **Rules for CLP-related parts:**
1. **Font-size requirement (minimum physical size):**
* Packaging ≤ 500 ml → **≥ 7.03 points (1.2 mm)**
* Packaging > 500 ml and ≤ 3000 ml → **≥ 8.20 points (1.4 mm)**
* Packaging > 3000 ml → **≥ 10.50 points (1.8 mm)**
* For inner packaging ≤ 10 ml → may be smaller, but must remain easily legible.
2. **Line-distance rule:**
Distance between two lines must be **≥ 120% of the font size**.
3. **Background rule:**
Background of all CLP text must be **white with black font**.
"""

In [ ]:
PROMPT_NON_CLP = f"""
You are an AI assistant that checks images of product labels 
### **Reference Scale:**
The image resolution is calculated as {TRUE_DPI} DPI ({DPmm:.2f} pixels per mm). Use this to verify font sizes.
### **Rules for non-CLP parts:**
1. Recommended minimum font size.
2. Recommended minimum line distance = 120% of font size.
3. Recommended background = white with black font.
"""

In [ ]:
def resize_image(img: PIL_Image.Image, width=None, height=None):
    """
    Resize a PIL image by specifying *either* width or height.
    Aspect ratio is preserved.
    """
    if width is None and height is None:
        raise ValueError("You must provide either width or height.")

    orig_w, orig_h = img.size

    if width is not None and height is not None:
        raise ValueError("Provide only width OR height, not both.")

    if width is not None:
        # scale using width
        scale = width / orig_w
        new_size = (width, int(orig_h * scale))

    if height is not None:
        # scale using height
        scale = height / orig_h
        new_size = (int(orig_w * scale), height)

    return img.resize(new_size, PIL_Image.LANCZOS)

In [ ]:
def pillow_to_base64(img, format="PNG"):
    buffer = BytesIO()
    img.save(buffer, format=format)
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return encoded

In [ ]:
for part in visual_check.parts:
    display(Markdown(f"### {part.classification} | {part.label}"))

    rect = part.rect
    abs_x1 = int(rect.xmin/1000 * width)
    abs_y1 = int(rect.ymin/1000 * height)
    abs_x2 = int(rect.xmax/1000 * width)
    abs_y2 = int(rect.ymax/1000 * height)
    
    cropped = img.crop(
        (abs_x1, abs_y1, abs_x2, abs_y2)
    )  
    cropped = resize_image (cropped, height=300)
    display(cropped)

    artwork_inline_data = {
        "inline_data": {
             "mime_type": "image/jpeg",
            "data": pillow_to_base64(cropped)
        }
    }
    
    response = client.models.generate_content(
      model=MODEL,
      contents=[
        PROMPT_CLP if part.classification == "CLP" else PROMPT_NON_CLP,
        artwork_inline_data
      ]
    )   
    display (Markdown(response.text))